In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 1. RECORDED TELEMETRY & FLIGHT DECISION ENGINE
# ==========================================
TELEMETRY_SCENARIOS = {
    "Battery Sag & Low Voltage": {
        "channel": "voltage_v",
        "ylabel": "Pack Voltage [V]",
        "default_thresh": 14.2,
        "trigger_when": "below",
        "decision": "TRIGGER RETURN-TO-HOME (RTH)",
        "badge_color": "#dc2626",
        "desc": "Recorded pack voltage drops under load and crosses the low-voltage reserve threshold before landing."
    },
    "Severe Wind Gust / G-Spike": {
        "channel": "accel_g",
        "ylabel": "Vertical Load [g]",
        "default_thresh": 2.10,
        "trigger_when": "above",
        "decision": "ABORT CLIMB & EASE G-LOAD",
        "badge_color": "#ea580c",
        "desc": "Recorded vertical acceleration spikes past the structural/comfort limit during a strong wind gust."
    },
    "Nominal Flight Envelope": {
        "channel": "voltage_v",
        "ylabel": "Pack Voltage [V]",
        "default_thresh": 14.2,
        "trigger_when": "below",
        "decision": "CONTINUE PLANNED MISSION",
        "badge_color": "#16a34a",
        "desc": "Recorded voltage, altitude, and g-load remain inside safe operating margins throughout the flight."
    },
}

def generate_recorded_telemetry(scenario_name, threshold_val, noise_level, duration_s=40.0, fs_hz=25.0, seed=303):
    prof = TELEMETRY_SCENARIOS[scenario_name]
    n = max(int(duration_s * fs_hz), 100)
    t = np.linspace(0.0, duration_s, n)
    u = t / max(duration_s, 1e-6)

    rng = np.random.default_rng(seed)
    noise_scale = max(noise_level, 0.0)

    # 2D flight profile: 0 m -> climb -> cruise -> descent
    alt_m = 65.0 * np.sin(np.pi * u) ** 0.75 + rng.normal(0.0, 0.4 * noise_scale, size=n)
    alt_m[0], alt_m[-1] = 0.0, 0.0
    dist_m = 12.0 * t

    # Recorded channels
    if scenario_name == "Battery Sag & Low Voltage":
        voltage_v = 16.4 - 2.9 * (u ** 1.25) - 0.55 * np.exp(-((u - 0.68) / 0.12) ** 2) + rng.normal(0.0, 0.05 * noise_scale, size=n)
        accel_g = 1.0 + 0.25 * np.sin(2.0 * np.pi * u) + rng.normal(0.0, 0.04 * noise_scale, size=n)
    elif scenario_name == "Severe Wind Gust / G-Spike":
        voltage_v = 16.4 - 1.3 * u + rng.normal(0.0, 0.04 * noise_scale, size=n)
        accel_g = 1.0 + 0.15 * np.sin(3.0 * np.pi * u) + 1.55 * np.exp(-((u - 0.54) / 0.05) ** 2) + rng.normal(0.0, 0.06 * noise_scale, size=n)
    else:  # Nominal Flight Envelope
        voltage_v = 16.4 - 1.45 * u + rng.normal(0.0, 0.04 * noise_scale, size=n)
        accel_g = 1.0 + 0.22 * np.sin(2.0 * np.pi * u) + rng.normal(0.0, 0.04 * noise_scale, size=n)

    df = pd.DataFrame({
        "time_s": t,
        "distance_m": dist_m,
        "altitude_m": alt_m,
        "voltage_v": voltage_v,
        "accel_g": accel_g,
    })

    series = df[prof["channel"]].to_numpy()
    if prof["trigger_when"] == "below":
        triggered_idx = np.where(series < threshold_val)[0]
    else:
        triggered_idx = np.where(series > threshold_val)[0]

    if len(triggered_idx) > 0:
        idx = int(triggered_idx[0])
        active_decision = prof["decision"]
        badge_color = prof["badge_color"]
        trigger_time = float(t[idx])
        trigger_val = float(series[idx])
    else:
        idx = int(np.argmax(alt_m))
        active_decision = "CONTINUE PLANNED MISSION"
        badge_color = "#16a34a"
        trigger_time = float(t[idx])
        trigger_val = float(series[idx])

    return df, {
        "idx": idx,
        "decision": active_decision,
        "badge_color": badge_color,
        "trigger_time": trigger_time,
        "trigger_val": trigger_val,
        "min_v": float(np.min(voltage_v)),
        "max_g": float(np.max(accel_g)),
    }

def build_decision_why(scenario_name, threshold_val, stats, prof):
    ch_label = prof["ylabel"]
    if stats["decision"] == "CONTINUE PLANNED MISSION":
        return (
            f"<b>Why This Decision Happened:</b> The dashboard highlights <b>CONTINUE PLANNED MISSION</b> because "
            f"recorded {ch_label} stayed within safe bounds (min voltage {stats['min_v']:.2f} V, max load {stats['max_g']:.2f} g) "
            f"relative to the {threshold_val:.2f} threshold."
        )
    direction_word = "dropped below" if prof["trigger_when"] == "below" else "exceeded"
    return (
        f"<b>Why This Decision Happened:</b> The dashboard highlights <b>{stats['decision']}</b> because "
        f"recorded {ch_label} {direction_word} the {threshold_val:.2f} threshold at <b>t = {stats['trigger_time']:.1f} s</b> "
        f"(recorded value: <b>{stats['trigger_val']:.2f}</b>), requiring immediate action rather than ignoring the trend."
    )

# ==========================================
# 2. DASHBOARD VISUALIZATION
# ==========================================
def draw_telemetry_dashboard(scenario_name, threshold_val, noise_level, run_idx):
    prof = TELEMETRY_SCENARIOS[scenario_name]
    df, stats = generate_recorded_telemetry(
        scenario_name, threshold_val, noise_level, seed=303 + run_idx
    )
    why_html = build_decision_why(scenario_name, threshold_val, stats, prof)

    plt.style.use("seaborn-v0_8-whitegrid")
    fig, (ax_2d, ax_tel) = plt.subplots(1, 2, figsize=(13.6, 4.6), gridspec_kw={"width_ratios": [1.05, 1.35]})

    # --- LEFT: 2D Flight Profile with Decision Point ---
    ax_2d.plot(df["distance_m"], df["altitude_m"], color="#2563eb", lw=2.2, label="Flight Path")
    ax_2d.fill_between(df["distance_m"], 0.0, df["altitude_m"], color="#93c5fd", alpha=0.25)
    ax_2d.scatter(
        [df["distance_m"].iloc[stats["idx"]]],
        [df["altitude_m"].iloc[stats["idx"]]],
        color=stats["badge_color"], s=85, zorder=7, label="Decision Point"
    )
    ax_2d.set_title("2D Flight Profile & Decision Point", fontsize=11, fontweight="bold", pad=10)
    ax_2d.set_xlabel("Downrange Distance [m]", fontsize=9.5)
    ax_2d.set_ylabel("Altitude [m]", fontsize=9.5)
    ax_2d.set_ylim(bottom=0.0)
    ax_2d.legend(loc="upper right", fontsize=8.3, frameon=True, facecolor="white")

    # --- RIGHT: Recorded Telemetry Channel & Threshold ---
    ch = prof["channel"]
    ax_tel.plot(df["time_s"], df[ch], color="#0284c7", lw=2.0, label=f"Recorded {prof['ylabel']}")
    ax_tel.axhline(threshold_val, color="#dc2626", ls="--", lw=1.6, label=f"Threshold ({threshold_val:.2f})")
    ax_tel.scatter([stats["trigger_time"]], [stats["trigger_val"]], color=stats["badge_color"], s=80, zorder=7)

    ax_tel.set_title(f"Telemetry Log — {scenario_name}", fontsize=11, fontweight="bold", pad=10)
    ax_tel.set_xlabel("Flight Time [s]", fontsize=9.5)
    ax_tel.set_ylabel(prof["ylabel"], fontsize=9.5)
    ax_tel.legend(loc="best", fontsize=8.3, frameon=True, facecolor="white")

    plt.tight_layout()
    plt.show()

    display(HTML(f"""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:5px solid {stats['badge_color']};
        border-radius:8px;padding:14px 18px;color:#f8fafc;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;margin-top:8px;
    ">
        <div style="display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:8px;">
            <div>
                <span style="font-size:11.5px;text-transform:uppercase;letter-spacing:0.6px;color:#94a3b8;">Highlighted Flight Decision</span><br/>
                <span style="font-size:16px;font-weight:800;color:#ffffff;">{stats['decision']}</span>
            </div>
            <div style="font-size:12.5px;color:#cbd5e1;">
                <b>Min Voltage:</b> {stats['min_v']:.2f} V &nbsp;|&nbsp;
                <b>Max Load:</b> {stats['max_g']:.2f} g &nbsp;|&nbsp;
                <b>Event Time:</b> {stats['trigger_time']:.1f} s
            </div>
        </div>
        <div style="font-size:12.5px;color:#e2e8f0;margin-top:8px;line-height:1.55;">
            {why_html}
        </div>
        <div style="font-size:12px;color:#94a3b8;margin-top:4px;line-height:1.45;">
            {prof['desc']}
        </div>
    </div>
    """))

# ==========================================
# 3. INTERACTIVE ENGLISH UI
# ==========================================
lbl_style = {"description_width": "155px"}
ctrl_layout = widgets.Layout(width="345px")

w_scen   = widgets.Dropdown(options=list(TELEMETRY_SCENARIOS.keys()), value="Battery Sag & Low Voltage", description="Telemetry Log:", style=lbl_style, layout=ctrl_layout)
w_thresh = widgets.FloatSlider(value=14.20, min=1.20, max=16.20, step=0.10, description="Decision Threshold:", readout_format=".2f", style=lbl_style, layout=ctrl_layout)
w_noise  = widgets.FloatSlider(value=0.8,   min=0.0,  max=3.0,   step=0.2,  description="Telemetry Noise:", readout_format=".1f", style=lbl_style, layout=ctrl_layout)

btn_dash = widgets.Button(description="Update Dashboard", button_style="primary", icon="tachometer", layout=widgets.Layout(width="210px", height="36px"))
out_dash = widgets.Output(layout=widgets.Layout(width="100%"))
dash_cnt = {"n": 0}
sync_t = {"flag": False}

def on_scenario_change(change):
    if sync_t["flag"]:
        return
    sync_t["flag"] = True
    w_thresh.value = TELEMETRY_SCENARIOS[change["new"]]["default_thresh"]
    sync_t["flag"] = False
    refresh_dash()

def refresh_dash(*args):
    if sync_t["flag"]:
        return
    dash_cnt["n"] += 1
    with out_dash:
        clear_output(wait=True)
        draw_telemetry_dashboard(w_scen.value, w_thresh.value, w_noise.value, dash_cnt["n"])

w_scen.observe(on_scenario_change, names="value")
btn_dash.on_click(refresh_dash)
for c in [w_thresh, w_noise]:
    c.observe(refresh_dash, names="value")

header_dash = widgets.HTML("""
    <div style="
        width:100%;box-sizing:border-box;
        background:linear-gradient(135deg,#0f172a 0%,#1e293b 100%);
        border:1px solid #334155;border-left:4px solid #38bdf8;
        border-radius:8px;padding:12px 16px;margin-bottom:10px;
        font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
    ">
        <div style="font-size:17px;font-weight:700;color:#f8fafc;">Flight Telemetry Decision Dashboard</div>
        <div style="font-size:12px;color:#94a3b8;margin-top:2px;">
            Inspect recorded sensor data and highlight one actionable flight decision with cause-and-effect reasoning
        </div>
    </div>
""")

controls_dash = widgets.VBox([
    widgets.HBox([
        widgets.VBox([w_scen, w_thresh], layout=widgets.Layout(margin="0 28px 0 0", gap="8px")),
        widgets.VBox([w_noise, btn_dash], layout=widgets.Layout(gap="8px"))
    ], layout=widgets.Layout(align_items="flex-start", flex_flow="row wrap"))
], layout=widgets.Layout(
    width="100%", padding="14px 16px", border="1px solid #334155",
    border_radius="8px", background_color="#1e293b", margin="0 0 12px 0"
))

display(header_dash, controls_dash, out_dash)
refresh_dash()

HTML(value='\n    <div style="\n        width:100%;box-sizing:border-box;\n        background:linear-gradient(…

Output(layout=Layout(width='100%'))